In [2]:
import requests
from bs4 import BeautifulSoup

def fetch_listing_html(url: str) -> str:
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                      "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
    }
    response = requests.get(url, headers=headers, timeout=15)
    response.raise_for_status()
    return response.text

# Paste a real Craigslist car listing URL to test
test_url = "https://www.craigslist.org/view/d/poughkeepsie-2003-ford-excursion/gUNKthbqotTAKUyZ8Dcr7y"
html = fetch_listing_html(test_url)
print(f"Fetched {len(html):,} characters")
print(html[:500])

Fetched 35,664 characters
<!DOCTYPE html>
<html>
<head>
    
	<meta charset="UTF-8">
	<meta http-equiv="X-UA-Compatible" content="IE=Edge">
	<meta name="viewport" content="width=device-width,initial-scale=1">
	<meta property="og:site_name" content="craigslist">
	<meta name="twitter:card" content="preview">
	<meta property="og:title" content="2003 ford excursion diesel for sale - Poughkeepsie, NY - craigslist">
	<meta name="description" content="2003 Ford Excursion Limited 6.0L V8 Turbo Diesel 4x4, Black with Tan leather.


In [3]:
def extract_visible_text(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")

    # Strip elements that won't contain useful listing text
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    text = soup.get_text(separator="\n")
    # Collapse excessive blank lines
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return "\n".join(lines)

listing_text = extract_visible_text(html)
print(f"Extracted {len(listing_text):,} characters of visible text")
print(listing_text[:1500])

Extracted 1,867 characters of visible text
2003 ford excursion diesel for sale - Poughkeepsie, NY - craigslist
CL
hudson valley
>
for sale by dealer
>
cars+trucks
post
account
favorites
hidden
CL
hudson valley            >
cars & trucks - by dealer
...
◀  prev
▲
next ▶
favorite
favorite
hide
unhide
⚐
⚑
flag
⚑
flagged
Posted
2026-07-10 12:57
Contact Information:
print
2003 Ford Excursion Limited 6.0L Turbo Diesel - *Very Clean* 4x4 SUV!!
-
$23,995
(Hyde Park/Poughkeepsie)
‹
image 1 of 15
›
120 E Dorsey Lane
2003
ford excursion
condition:
excellent
cylinders:
8 cylinders
drive:
4wd
fuel:
diesel
odometer:
151,000
paint color:
black
title status:
clean
transmission:
automatic
type:
SUV
more ads by this seller
QR Code Link to This Post
2003 Ford Excursion Limited 6.0L V8 Turbo Diesel 4x4, Black with Tan leather. This 7-Passenger Diesel SUV is *very clean* and runs and drives excellent! It has 151,000 miles and is equipped with a Clean Carfax, power windows and locks, power leather heated se

In [4]:
from pydantic import BaseModel, Field
from typing import Optional
from litellm import completion
import json

class ExtractedListing(BaseModel):
    title: str = Field(description="Full listing title, e.g. '2003 Ford Excursion Limited'")
    make: str
    model: str
    trim: Optional[str] = None
    year: int
    mileage: float = Field(description="Odometer reading in miles")
    price: float = Field(description="Asking price in dollars, numeric only")
    horsepower: Optional[float] = None
    body_type: Optional[str] = None
    fuel_type: Optional[str] = None
    transmission: Optional[str] = None
    has_accidents: Optional[bool] = None
    frame_damaged: Optional[bool] = None
    summary: str = Field(description="1-3 sentence plain description of the car's condition and features")

def extract_car_details(listing_text: str) -> ExtractedListing:
    prompt = f"""Extract structured car listing data from this text. Return ONLY valid JSON matching this schema, no explanation:

{{
  "title": "...", "make": "...", "model": "...", "trim": "...", "year": ...,
  "mileage": ..., "price": ..., "horsepower": ..., "body_type": "...",
  "fuel_type": "...", "transmission": "...", "has_accidents": ..., "frame_damaged": ...,
  "summary": "..."
}}

Use null for fields you cannot confidently determine. Do not guess has_accidents or frame_damaged unless the text explicitly mentions accident/damage history.

Listing text:
{listing_text}"""

    response = completion(
        model="gemini/gemini-2.5-flash",
        messages=[{"role": "user", "content": prompt}]
    )
    raw = response.choices[0].message.content.strip()
    raw = raw.replace("```json", "").replace("```", "").strip()
    data = json.loads(raw)
    return ExtractedListing(**data)

extracted = extract_car_details(listing_text)
print(extracted.model_dump_json(indent=2))

{
  "title": "2003 Ford Excursion Limited 6.0L Turbo Diesel - *Very Clean* 4x4 SUV!!",
  "make": "Ford",
  "model": "Excursion",
  "trim": "Limited",
  "year": 2003,
  "mileage": 151000.0,
  "price": 23995.0,
  "horsepower": null,
  "body_type": "SUV",
  "fuel_type": "Diesel",
  "transmission": "automatic",
  "has_accidents": false,
  "frame_damaged": false,
  "summary": "2003 Ford Excursion Limited 6.0L V8 Turbo Diesel 4x4, black with tan leather. This 7-passenger SUV is very clean and runs and drives excellent with 151,000 miles. It is equipped with a Clean Carfax, power windows and locks, power leather heated seats, captain chairs in second row, and 2 keys. The title status is clean."
}


In [5]:
def extract_car_details(listing_text: str) -> ExtractedListing:
    prompt = f"""Extract structured car listing data from this text. Return ONLY valid JSON matching this schema, no explanation:

{{
  "title": "...", "make": "...", "model": "...", "trim": "...", "year": ...,
  "mileage": ..., "price": ..., "horsepower": ..., "body_type": "...",
  "fuel_type": "...", "transmission": "...", "has_accidents": ..., "frame_damaged": ...,
  "summary": "..."
}}

For has_accidents and frame_damaged: use null unless the text EXPLICITLY states
an accident or frame damage occurred (e.g. "was in an accident", "frame damage",
"salvage title"). Do NOT infer false from phrases like "clean Carfax", "clean title",
or "no issues" — those are not the same claim as "confirmed no accidents". Only set
these to true or false when the text directly addresses accident/damage history.

Listing text:
{listing_text}"""

    response = completion(
        model="gemini/gemini-2.5-flash",
        messages=[{"role": "user", "content": prompt}]
    )
    raw = response.choices[0].message.content.strip()
    raw = raw.replace("```json", "").replace("```", "").strip()
    data = json.loads(raw)
    return ExtractedListing(**data)

extracted = extract_car_details(listing_text)
print(extracted.model_dump_json(indent=2))

{
  "title": "2003 Ford Excursion Limited 6.0L Turbo Diesel - *Very Clean* 4x4 SUV!!",
  "make": "Ford",
  "model": "Excursion",
  "trim": "Limited 6.0L Turbo Diesel",
  "year": 2003,
  "mileage": 151000.0,
  "price": 23995.0,
  "horsepower": null,
  "body_type": "SUV",
  "fuel_type": "diesel",
  "transmission": "automatic",
  "has_accidents": null,
  "frame_damaged": null,
  "summary": "This 7-Passenger Diesel SUV in black with tan leather is very clean and runs and drives excellent! It has 151,000 miles and is equipped with power windows and locks, power leather heated seats, captain chairs in the second row, and 2 keys. Comes with a Clean Carfax report. No hidden dealer fees. Extended warranties and financing options are available."
}


In [6]:
import sys
sys.path.append('..')
from auto_pricer.car import Car

def to_car(extracted: ExtractedListing) -> Car:
    return Car(
        title=extracted.title,
        make=extracted.make,
        model=extracted.model,
        trim=extracted.trim,
        year=extracted.year,
        mileage=extracted.mileage,
        price=extracted.price,
        horsepower=extracted.horsepower,
        body_type=extracted.body_type,
        fuel_type=extracted.fuel_type,
        transmission=extracted.transmission,
        has_accidents=extracted.has_accidents,
        frame_damaged=extracted.frame_damaged,
        summary=extracted.summary,
    )

car = to_car(extracted)
print(car)
print(f"\nSummary used for pricing:\n{car.summary}")

title='2003 Ford Excursion Limited 6.0L Turbo Diesel - *Very Clean* 4x4 SUV!!' make='Ford' model='Excursion' trim='Limited 6.0L Turbo Diesel' year=2003 mileage=151000.0 price=23995.0 horsepower=None body_type='SUV' fuel_type='diesel' transmission='automatic' has_accidents=None frame_damaged=None full=None summary='This 7-Passenger Diesel SUV in black with tan leather is very clean and runs and drives excellent! It has 151,000 miles and is equipped with power windows and locks, power leather heated seats, captain chairs in the second row, and 2 keys. Comes with a Clean Carfax report. No hidden dealer fees. Extended warranties and financing options are available.' prompt=None completion=None id=None

Summary used for pricing:
This 7-Passenger Diesel SUV in black with tan leather is very clean and runs and drives excellent! It has 151,000 miles and is equipped with power windows and locks, power leather heated seats, captain chairs in the second row, and 2 keys. Comes with a Clean Carfax 

In [7]:
from auto_pricer.agents import EnsembleAgent

ensemble = EnsembleAgent()
predicted_price = ensemble.price(car.summary)

print(f"Scraped listing price (seller's asking price): ${car.price:,.0f}")
print(f"EnsembleAgent predicted price: ${predicted_price:,.0f}")
print(f"Difference: ${predicted_price - car.price:,.0f}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Scraped listing price (seller's asking price): $23,995
EnsembleAgent predicted price: $17,498
Difference: $-6,498


In [8]:
from auto_pricer.agents import SpecialistAgent, FrontierAgent

specialist = SpecialistAgent()
frontier = FrontierAgent()

p_specialist = specialist.price(car.summary)
p_frontier = frontier.price(car.summary)

print(f"Asking price: ${car.price:,.0f}")
print(f"Specialist (fine-tuned, no RAG): ${p_specialist:,.0f}")
print(f"Frontier (Gemini + RAG): ${p_frontier:,.0f}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Asking price: $23,995
Specialist (fine-tuned, no RAG): $13,995
Frontier (Gemini + RAG): $16,995


In [9]:
p_frontier_1 = frontier.price(car.summary)
p_frontier_2 = frontier.price(car.summary)
print(f"Frontier call 1: ${p_frontier_1:,.0f}")
print(f"Frontier call 2: ${p_frontier_2:,.0f}")

Frontier call 1: $16,495
Frontier call 2: $14,450


In [10]:
docs, prices = frontier._find_similars(car.summary)
for doc, price in zip(docs, prices):
    print(f"${price:,.0f} — {doc[:100]}")

$32,995 — Title: 2016 Lexus RX 350 AWD
Category: SUV
Make: Lexus
Description: This one-owner Lexus RX 350 is i
$32,188 — Title: 2019 Acura RDX FWD
Category: SUV
Make: Acura
Description: One owner, non-smoker vehicle in ex
$15,772 — Title: 2013 Lexus RX 350 AWD SUV
Category: SUV
Make: Lexus
Description: This well-maintained 2013 Le
$17,444 — Title: 2014 Lexus RX 350 SUV
Category: SUV
Make: Lexus
Description: This one-owner, clean Carfax RX 
$18,495 — Title: 2013 Lexus RX 350 FWD
Category: SUV
Make: Lexus
Description: This one-owner 2013 Lexus RX 350


In [12]:
import sys
sys.path.append('..')
from auto_pricer.agents import FrontierAgent

frontier = FrontierAgent()
p1 = frontier.price(car.summary)
p2 = frontier.price(car.summary)
print(f"Frontier call 1: ${p1:,.0f}")
print(f"Frontier call 2: ${p2:,.0f}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Frontier call 1: $16,995
Frontier call 2: $14,995


In [16]:
import asyncio
import sys

if sys.platform == "win32":
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

from concurrent.futures import ThreadPoolExecutor
from playwright.sync_api import sync_playwright

def run_smoke_test():
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        page = browser.new_page()
        page.goto("https://example.com")
        title = page.title()
        browser.close()
        return title

with ThreadPoolExecutor(1) as executor:
    result = executor.submit(run_smoke_test).result()

print(result)

Example Domain


In [17]:
def fetch_fb_listing(url: str) -> str:
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        page = browser.new_page()
        page.goto(url, timeout=30000)
        page.wait_for_timeout(3000)  # let JS render
        html = page.content()
        browser.close()
        return html

with ThreadPoolExecutor(1) as executor:
    fb_html = executor.submit(fetch_fb_listing, "https://www.facebook.com/marketplace/item/1524895609094527/?ref=browse_tab&referral_code=marketplace_top_picks&referral_story_type=top_picks").result()

print(f"Fetched {len(fb_html):,} characters")
print(fb_html[:1000])

Fetched 1,219,719 characters
<!DOCTYPE html><html id="facebook" class="_9dls __fb-light-mode" lang="en" dir="ltr"><head><link data-default-icon="https://static.xx.fbcdn.net/rsrc.php/y1/r/ay1hV6OlegS.ico" data-badged-icon="https://static.xx.fbcdn.net/rsrc.php/yD/r/UJj0tgk-RrT.ico" rel="shortcut icon" href="https://static.xx.fbcdn.net/rsrc.php/y1/r/ay1hV6OlegS.ico"><meta name="bingbot" content="noarchive"><meta name="viewport" content="width=device-width,initial-scale=1,maximum-scale=2,shrink-to-fit=no"><meta property="al:android:app_name" content="Facebook"><meta property="al:android:package" content="com.facebook.katana"><meta property="al:android:url" content="fb://marketplace_product_details/?id=1524895609094527"><meta property="al:ios:app_name" content="Facebook"><meta property="al:ios:app_store_id" content="284882215"><meta property="al:ios:url" content="fb://marketplace_product_details/?id=1524895609094527"><meta name="apple-itunes-app" content="app-id=284882215, app-argument=fb:/

In [18]:
# Check for login-wall indicators
login_signals = ["Log in to Facebook", "You must log in", "log in or sign up", "Log In"]
found_signals = [s for s in login_signals if s.lower() in fb_html.lower()]
print(f"Login-wall signals found: {found_signals}")

# Extract visible text the same way as Craigslist
def extract_visible_text(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    text = soup.get_text(separator="\n")
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return "\n".join(lines)

fb_text = extract_visible_text(fb_html)
print(f"\nExtracted {len(fb_text):,} characters of visible text")
print(fb_text[:1500])

Login-wall signals found: ['Log in to Facebook', 'Log In']

Extracted 2,881 characters of visible text
2011 BMW 328i xdrive - Cars & Trucks - Florida, New York | Facebook Marketplace | Facebook
Facebook
Log In
Log In
Forgot Account?
Marketplace
Browse all
Jobs
Your account
Create new listing
Location
Florida, New York
·
Within 40 mi
Categories
Vehicles
Property Rentals
Apparel
Classifieds
Electronics
Entertainment
Family
Free Stuff
Garden & Outdoor
Hobbies
Home Goods
Home Improvement Supplies
Home Sales
Musical Instruments
Office Supplies
Pet Supplies
Sporting Goods
Toys & Games
More categories
Nearby Cities
Chester, New York
Warwick, New York
Goshen, New York
New Hampton, New York
Pine Island, New York
See more
0:03
/
0:14
2011 BMW 328i xdrive
$5,000
Listed
3 weeks ago
in
Florida, NY
Message
Message
Save
Share
Save
Share
Details
Condition
Used - Good
153xxx miles
Car needs nothing. Exterior is 8/10 Interior is 9/10. All wheel drive. Zero warning lights on the dash, all services up to 

In [19]:
extracted_fb = extract_car_details(fb_text)
print(extracted_fb.model_dump_json(indent=2))

{
  "title": "2011 BMW 328i xdrive",
  "make": "BMW",
  "model": "328i",
  "trim": "xdrive",
  "year": 2011,
  "mileage": 153000.0,
  "price": 5000.0,
  "horsepower": null,
  "body_type": null,
  "fuel_type": null,
  "transmission": null,
  "has_accidents": null,
  "frame_damaged": null,
  "summary": "Condition Used - Good. 153xxx miles. Car needs nothing. Exterior is 8/10 Interior is 9/10. All wheel drive. Zero warning lights on the dash, all services up to date. TPMS system fully working. 4 matching good tires. Active headlights (turn left and right with steering wheel). Heated leather seats and heated steering wheel. Radio with factory subs under the seats work perfectly. Power everything and sun/moon roof. Work done within the last year: Front lower/rearward control arms, Rear shocks, Front right CX axle, Both front wheel bearings, Brakes and sensors all around, Alignment, Oil changed every 10k with full synthetic and OEM filter. Great daily driver. Seller is looking at another car

In [20]:
car_fb = to_car(extracted_fb)
predicted_price_fb = ensemble.price(car_fb.summary)

print(f"Asking price: ${car_fb.price:,.0f}")
print(f"EnsembleAgent predicted price: ${predicted_price_fb:,.0f}")

Asking price: $5,000
EnsembleAgent predicted price: $14,872


In [21]:
def format_summary_for_pricing(extracted: ExtractedListing) -> str:
    category = extracted.body_type or "Vehicle"
    details_parts = [f"{extracted.mileage:,.0f} miles"]
    if extracted.fuel_type:
        details_parts.append(extracted.fuel_type)
    if extracted.transmission:
        details_parts.append(extracted.transmission)
    details = ", ".join(details_parts)

    return (
        f"Title: {extracted.year} {extracted.make} {extracted.model} {extracted.trim or ''}".strip() + "\n"
        f"Category: {category}\n"
        f"Make: {extracted.make}\n"
        f"Description: {extracted.summary}\n"
        f"Details: {details}."
    )

# Rebuild both test cars with the corrected summary format
car_fb.summary = format_summary_for_pricing(extracted_fb)
print(car_fb.summary)

Title: 2011 BMW 328i xdrive
Category: Vehicle
Make: BMW
Description: Condition Used - Good. 153xxx miles. Car needs nothing. Exterior is 8/10 Interior is 9/10. All wheel drive. Zero warning lights on the dash, all services up to date. TPMS system fully working. 4 matching good tires. Active headlights (turn left and right with steering wheel). Heated leather seats and heated steering wheel. Radio with factory subs under the seats work perfectly. Power everything and sun/moon roof. Work done within the last year: Front lower/rearward control arms, Rear shocks, Front right CX axle, Both front wheel bearings, Brakes and sensors all around, Alignment, Oil changed every 10k with full synthetic and OEM filter. Great daily driver. Seller is looking at another car, which is the only reason for selling. One wheel doesn't match, seller has the matching one but it needs to be repaired.
Details: 153,000 miles.


In [22]:
predicted_price_fb_v2 = ensemble.price(car_fb.summary)
print(f"Asking price: ${car_fb.price:,.0f}")
print(f"EnsembleAgent predicted price (v1, prose summary): $14,872")
print(f"EnsembleAgent predicted price (v2, structured template): ${predicted_price_fb_v2:,.0f}")

Asking price: $5,000
EnsembleAgent predicted price (v1, prose summary): $14,872
EnsembleAgent predicted price (v2, structured template): $9,122


In [24]:
import pandas as pd

df = pd.read_csv("../data/used_cars_data.csv")  # replace with actual filename
print(f"Rows in raw CSV: {len(df):,}")
print(f"Rows currently in training set: {len(train):,}")
print(f"\nColumns: {list(df.columns)}")
df.head(3)

C:\Users\mrsha\AppData\Local\Temp\ipykernel_20268\1557937883.py:3: DtypeWarning: Columns (0: dealer_zip) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/used_cars_data.csv")  # replace with actual filename


Rows in raw CSV: 3,000,040


NameError: name 'train' is not defined

In [25]:
print(df.columns.tolist())
print(f"\nNull counts:\n{df.isnull().sum().sort_values(ascending=False).head(15)}")

['vin', 'back_legroom', 'bed', 'bed_height', 'bed_length', 'body_type', 'cabin', 'city', 'city_fuel_economy', 'combine_fuel_economy', 'daysonmarket', 'dealer_zip', 'description', 'engine_cylinders', 'engine_displacement', 'engine_type', 'exterior_color', 'fleet', 'frame_damaged', 'franchise_dealer', 'franchise_make', 'front_legroom', 'fuel_tank_volume', 'fuel_type', 'has_accidents', 'height', 'highway_fuel_economy', 'horsepower', 'interior_color', 'isCab', 'is_certified', 'is_cpo', 'is_new', 'is_oemcpo', 'latitude', 'length', 'listed_date', 'listing_color', 'listing_id', 'longitude', 'main_picture_url', 'major_options', 'make_name', 'maximum_seating', 'mileage', 'model_name', 'owner_count', 'power', 'price', 'salvage', 'savings_amount', 'seller_rating', 'sp_id', 'sp_name', 'theft_title', 'torque', 'transmission', 'transmission_display', 'trimId', 'trim_name', 'vehicle_damage_category', 'wheel_system', 'wheel_system_display', 'wheelbase', 'width', 'year']

Null counts:
is_certified     

In [26]:
df_full = pd.read_csv("../data/used_cars_data.csv", nrows=500_000)

C:\Users\mrsha\AppData\Local\Temp\ipykernel_20268\1117102834.py:1: DtypeWarning: Columns (0: bed, 1: dealer_zip) have mixed types. Specify dtype option on import or set low_memory=False.
  df_full = pd.read_csv("../data/used_cars_data.csv", nrows=500_000)


In [27]:
df_new = df.iloc[500_000:700_000].copy()
print(f"New raw chunk: {len(df_new):,} rows")
print(df_new[['make_name', 'model_name', 'year', 'price', 'mileage']].head(3))

New raw chunk: 200,000 rows
       make_name model_name  year    price  mileage
500000    Toyota    4Runner  2017  30495.0  62948.0
500001    Toyota       RAV4  2012  12495.0  92700.0
500002     Lexus     RX 350  2017  34995.0  32550.0


In [30]:
import numpy as np

# Step 1 — drop dead/junk columns
dead_columns = [
    'is_certified', 'combine_fuel_economy', 'vehicle_damage_category',
    'bed', 'cabin', 'bed_length', 'bed_height', 'is_oemcpo', 'isCab',
    'main_picture_url', 'listing_id', 'sp_id', 'sp_name', 'dealer_zip',
    'latitude', 'longitude', 'vin'
]
df_clean = df_new.drop(columns=[c for c in dead_columns if c in df_new.columns])

# Step 2 — drop is_new == True (used-car project only)
df_clean = df_clean[df_clean['is_new'] != True]

# Step 3 — impute damage-history cluster with False, owner_count with median
damage_cluster = ['salvage', 'theft_title', 'frame_damaged', 'fleet', 'has_accidents']
for col in damage_cluster:
    df_clean[col] = df_clean[col].fillna(False)
df_clean['owner_count'] = df_clean['owner_count'].fillna(df_clean['owner_count'].median())

# Step 4 — impute moderate-missingness numerics with median (torque/power excluded: not in Car schema, stored as compound strings)
moderate_numeric = ['horsepower', 'highway_fuel_economy', 'city_fuel_economy', 'engine_displacement']
for col in moderate_numeric:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Step 5 — drop rows missing required fields
required = ['price', 'mileage', 'year', 'make_name', 'model_name']
df_clean = df_clean.dropna(subset=required)

# Step 6 — threshold filters
df_clean = df_clean[
    df_clean['price'].between(500, 150_000) &
    (df_clean['year'] >= 1990) &
    (df_clean['mileage'] <= 300_000)
]

print(f"Started with: {len(df_new):,} raw rows")
print(f"Survived cleaning: {len(df_clean):,} rows")
print(f"Survival rate: {len(df_clean) / len(df_new):.1%}")

Started with: 200,000 raw rows
Survived cleaning: 100,071 rows
Survival rate: 50.0%


In [29]:
print(df_clean[['horsepower', 'torque', 'power', 'highway_fuel_economy', 'city_fuel_economy', 'engine_displacement']].dtypes)
print()
print(df_clean[['torque', 'power']].head(3))

horsepower              float64
torque                      str
power                       str
highway_fuel_economy    float64
city_fuel_economy       float64
engine_displacement     float64
dtype: object

                       torque               power
500000  278 lb-ft @ 4,400 RPM  270 hp @ 5,600 RPM
500001  172 lb-ft @ 4,000 RPM  179 hp @ 6,000 RPM
500002                    NaN                 NaN


In [32]:
import sys
sys.path.append('..')
from auto_pricer.car import Car

train, val, test = Car.from_hub("ShahidHKhan/cars_full_processed")
print(f"Loaded {len(train):,} training cars")

Future exception was never retrieved
future: <Future finished exception=NotImplementedError()>
Traceback (most recent call last):
  File "d:\mrsha\Projects\AutoPricer\.venv\Lib\site-packages\IPython\core\interactiveshell.py", line 3748, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "C:\Users\mrsha\AppData\Local\Temp\ipykernel_20268\3490229128.py", line 14, in <module>
    result = executor.submit(run_smoke_test).result()
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\mrsha\AppData\Local\Programs\Python\Python312\Lib\concurrent\futures\_base.py", line 456, in result
    return self.__get_result()
           ^^^^^^^^^^^^^^^^^^^
  File "C:\Users\mrsha\AppData\Local\Programs\Python\Python312\Lib\concurrent\futures\_base.py", line 401, in __get_result
    raise self._exception
  File "C:\Users\mrsha\AppData\Local\Programs\Python\Python312\Lib\concurrent\futures\thread.py", line 58, in run
    result = self.fn(*self.args, **self.kwargs)
    

Loaded 231,947 training cars


In [34]:
# Trim to exactly what we need — reproducible with a fixed seed
df_final = df_clean.sample(n=68_053, random_state=42).reset_index(drop=True)
print(f"Trimmed to: {len(df_final):,} rows")

def build_full_text(row) -> str:
    desc = str(row.get('description', '') or '')[:400]  # cap ~400 chars
    return (
        f"{row['year']} {row['make_name']} {row['model_name']} {row.get('trim_name', '') or ''}\n"
        f"Body: {row.get('body_type', 'Unknown')}\n"
        f"Mileage: {row['mileage']:,.0f}\n"
        f"Horsepower: {row.get('horsepower', 'Unknown')}\n"
        f"Fuel: {row.get('fuel_type', 'Unknown')}\n"
        f"Transmission: {row.get('transmission', 'Unknown')}\n"
        f"Accidents: {row.get('has_accidents', False)}\n"
        f"Frame damaged: {row.get('frame_damaged', False)}\n"
        f"Description: {desc}"
    ).strip()

new_cars = []
next_id = len(train) + len(val) + len(test)  # continue id sequence from existing data

for _, row in df_final.iterrows():
    car = Car(
        title=f"{int(row['year'])} {row['make_name']} {row['model_name']}",
        make=row['make_name'],
        model=row['model_name'],
        trim=row.get('trim_name'),
        year=int(row['year']),
        mileage=float(row['mileage']),
        price=float(row['price']),
        horsepower=row.get('horsepower'),
        body_type=row.get('body_type'),
        fuel_type=row.get('fuel_type'),
        transmission=row.get('transmission'),
        has_accidents=row.get('has_accidents'),
        frame_damaged=row.get('frame_damaged'),
        full=build_full_text(row),
        id=next_id,
    )
    new_cars.append(car)
    next_id += 1

print(f"Built {len(new_cars):,} Car objects")
print(new_cars[0])
print(f"\nFull text sample:\n{new_cars[0].full}")

Trimmed to: 68,053 rows


ValidationError: 1 validation error for Car
fuel_type
  Input should be a valid string [type=string_type, input_value=nan, input_type=float]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type

In [35]:
import pandas as pd

def clean_val(v):
    return None if pd.isna(v) else v

new_cars = []
next_id = len(train) + len(val) + len(test)

for _, row in df_final.iterrows():
    car = Car(
        title=f"{int(row['year'])} {row['make_name']} {row['model_name']}",
        make=row['make_name'],
        model=row['model_name'],
        trim=clean_val(row.get('trim_name')),
        year=int(row['year']),
        mileage=float(row['mileage']),
        price=float(row['price']),
        horsepower=clean_val(row.get('horsepower')),
        body_type=clean_val(row.get('body_type')),
        fuel_type=clean_val(row.get('fuel_type')),
        transmission=clean_val(row.get('transmission')),
        has_accidents=clean_val(row.get('has_accidents')),
        frame_damaged=clean_val(row.get('frame_damaged')),
        full=build_full_text(row),
        id=next_id,
    )
    new_cars.append(car)
    next_id += 1

print(f"Built {len(new_cars):,} Car objects")
print(new_cars[0])
print(f"\nFull text sample:\n{new_cars[0].full}")

Built 68,053 Car objects
title='2017 Audi A4' make='Audi' model='A4' trim='2.0T quattro Premium Sedan AWD' year=2017 mileage=46060.0 price=25995.0 horsepower=252.0 body_type='Sedan' fuel_type='Gasoline' transmission='A' has_accidents=False frame_damaged=False full='2017 Audi A4 2.0T quattro Premium Sedan AWD\nBody: Sedan\nMileage: 46,060\nHorsepower: 252.0\nFuel: Gasoline\nTransmission: A\nAccidents: False\nFrame damaged: False\nDescription: [!@@Additional Info@@!]Power Front Bucket Seats,Leather Seating Surfaces,Radio: Audi Sound System,4-Wheel Disc Brakes,Air Conditioning,Electronic Stability Control,Front Bucket Seats,Front Center Armrest,Leather Shift Knob,Tachometer,ABS brakes,AM/FM radio,Alloy wheels,Anti-whiplash front head restraints,Automatic temperature control,Brake assist,Bumpers: body-color,CD player,Delay-off headlights,' summary=None prompt=None completion=None id=257719

Full text sample:
2017 Audi A4 2.0T quattro Premium Sedan AWD
Body: Sedan
Mileage: 46,060
Horsepower

In [36]:
import re

def scrub_description(desc: str) -> str:
    if not desc or pd.isna(desc):
        return ""
    desc = re.sub(r'\[!@@.*?@@!\]', '', desc)  # strip [!@@...@@!] markers
    desc = re.sub(r'\s*,\s*', ', ', desc)  # normalize comma spacing
    return desc.strip()[:400]

def build_full_text(row) -> str:
    desc = scrub_description(str(row.get('description', '') or ''))
    return (
        f"{row['year']} {row['make_name']} {row['model_name']} {row.get('trim_name', '') or ''}\n"
        f"Body: {row.get('body_type', 'Unknown')}\n"
        f"Mileage: {row['mileage']:,.0f}\n"
        f"Horsepower: {row.get('horsepower', 'Unknown')}\n"
        f"Fuel: {row.get('fuel_type', 'Unknown')}\n"
        f"Transmission: {row.get('transmission_display', 'Unknown')}\n"
        f"Accidents: {row.get('has_accidents', False)}\n"
        f"Frame damaged: {row.get('frame_damaged', False)}\n"
        f"Description: {desc}"
    ).strip()

new_cars = []
next_id = len(train) + len(val) + len(test)

for _, row in df_final.iterrows():
    car = Car(
        title=f"{int(row['year'])} {row['make_name']} {row['model_name']}",
        make=row['make_name'],
        model=row['model_name'],
        trim=clean_val(row.get('trim_name')),
        year=int(row['year']),
        mileage=float(row['mileage']),
        price=float(row['price']),
        horsepower=clean_val(row.get('horsepower')),
        body_type=clean_val(row.get('body_type')),
        fuel_type=clean_val(row.get('fuel_type')),
        transmission=clean_val(row.get('transmission_display')),
        has_accidents=clean_val(row.get('has_accidents')),
        frame_damaged=clean_val(row.get('frame_damaged')),
        full=build_full_text(row),
        id=next_id,
    )
    new_cars.append(car)
    next_id += 1

print(f"Built {len(new_cars):,} Car objects")
print(new_cars[0])
print(f"\nFull text sample:\n{new_cars[0].full}")

Built 68,053 Car objects
title='2017 Audi A4' make='Audi' model='A4' trim='2.0T quattro Premium Sedan AWD' year=2017 mileage=46060.0 price=25995.0 horsepower=252.0 body_type='Sedan' fuel_type='Gasoline' transmission='Automatic' has_accidents=False frame_damaged=False full='2017 Audi A4 2.0T quattro Premium Sedan AWD\nBody: Sedan\nMileage: 46,060\nHorsepower: 252.0\nFuel: Gasoline\nTransmission: Automatic\nAccidents: False\nFrame damaged: False\nDescription: Power Front Bucket Seats, Leather Seating Surfaces, Radio: Audi Sound System, 4-Wheel Disc Brakes, Air Conditioning, Electronic Stability Control, Front Bucket Seats, Front Center Armrest, Leather Shift Knob, Tachometer, ABS brakes, AM/FM radio, Alloy wheels, Anti-whiplash front head restraints, Automatic temperature control, Brake assist, Bumpers: body-color, CD player, Delay-off headlights, Driv' summary=None prompt=None completion=None id=257719

Full text sample:
2017 Audi A4 2.0T quattro Premium Sedan AWD
Body: Sedan
Mileage: 4

In [37]:
import time, random, litellm
from litellm import completion
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

litellm.suppress_debug_info = True

SYSTEM_PROMPT = """Create a concise description of a used car listing. Respond only in this format.
Title: Rewritten short precise title (year make model trim)
Category: e.g. Sedan, SUV, Truck
Make: Brand name
Description: 1 sentence on condition/history
Details: 1 sentence on mileage, fuel type, transmission"""

class Preprocessor:
    def __init__(self, model_name="gemini/gemini-2.5-flash-lite", max_retries=5):
        self.model_name = model_name
        self.max_retries = max_retries
        self.total_cost = 0

    def messages_for(self, text):
        return [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": text}
        ]

    def preprocess(self, car):
        messages = self.messages_for(car.full)
        for attempt in range(self.max_retries):
            try:
                response = completion(model=self.model_name, messages=messages, timeout=15)
                self.total_cost += response._hidden_params["response_cost"]
                car.summary = response.choices[0].message.content
                return car
            except Exception:
                wait = min((2 ** attempt) + random.uniform(0, 1), 10)
                time.sleep(wait)
        return car  # leaves car.summary = None on failure

# --- 5-call latency diagnostic first, per your doc's own lesson from the 503 outage ---
preprocessor = Preprocessor()
print("Running 5-call sequential diagnostic...")
for i in range(5):
    start = time.time()
    result = preprocessor.preprocess(new_cars[i])
    elapsed = time.time() - start
    print(f"Call {i+1}: {elapsed:.2f}s — {'OK' if result.summary else 'FAILED'}")

Running 5-call sequential diagnostic...
Call 1: 1.06s — OK
Call 2: 0.66s — OK
Call 3: 0.64s — OK
Call 4: 0.67s — OK
Call 5: 0.80s — OK


In [38]:
def run_preprocessing_batch(preprocessor, cars, max_workers=10):
    pbar = tqdm(total=len(cars))
    failures = [0]

    def run_one(car):
        result = preprocessor.preprocess(car)
        if result.summary is None:
            failures[0] += 1
        pbar.update(1)
        pbar.set_postfix(failures=failures[0])
        return result

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = [ex.submit(run_one, car) for car in cars]
        for f in as_completed(futures):
            f.result()
    pbar.close()
    print(f"Total cost: ${preprocessor.total_cost:.4f}")
    return cars

preprocessor = Preprocessor()
new_cars = run_preprocessing_batch(preprocessor, new_cars, max_workers=10)

succeeded = sum(1 for c in new_cars if c.summary is not None)
print(f"\nSucceeded: {succeeded:,}/{len(new_cars):,}")

100%|██████████| 68053/68053 [1:03:44<00:00, 17.79it/s, failures=0]


Total cost: $3.4149

Succeeded: 68,053/68,053


In [39]:
for car in new_cars[:3]:
    print(car.summary)
    print("---")

Title: 2017 Audi A4 2.0T quattro Premium Sedan
Category: Sedan
Make: Audi
Description: Well-maintained, one-owner Audi A4 with no reported accidents or frame damage.
Details: 46,060 miles, Gasoline, Automatic transmission.
---
Title: 2018 Ram 1500 Big Horn Crew Cab 4WD
Category: Truck
Make: Ram
Description: Certified pre-owned, one-owner Ram 1500 Big Horn in excellent condition with desirable options.
Details: 21,600 miles, Flex Fuel, Automatic transmission.
---
Title: 2015 GMC Acadia SLT-1 AWD SUV
Category: SUV
Make: GMC
Description: This 2015 GMC Acadia SLT-1 AWD is a well-maintained, one-owner vehicle with a clean title and no reported accidents.
Details: With 80,146 miles, this gasoline-powered SUV features an automatic transmission and comfortable seating for up to seven passengers.
---


In [41]:
import chromadb
from sentence_transformers import SentenceTransformer

client = chromadb.PersistentClient(path="../cars_vectorstore")
collection = client.get_or_create_collection("cars")
encoder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

print(f"Collection currently has: {collection.count():,} entries")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Collection currently has: 0 entries


In [42]:
BATCH_SIZE = 1000
existing_count = collection.count()
print(f"Collection currently has: {existing_count:,} entries")

for i in tqdm(range(0, len(new_cars), BATCH_SIZE)):
    batch = new_cars[i:i + BATCH_SIZE]
    documents = [car.summary for car in batch]
    vectors = encoder.encode(documents).astype(float).tolist()
    metadatas = [{"make": car.make, "year": car.year, "price": car.price} for car in batch]
    ids = [f"car_new_{i + j}" for j in range(len(batch))]
    collection.add(ids=ids, documents=documents, embeddings=vectors, metadatas=metadatas)

print(f"Collection now has: {collection.count():,} entries")

Collection currently has: 0 entries


100%|██████████| 69/69 [08:33<00:00,  7.44s/it]

Collection now has: 68,053 entries


In [43]:
import os
print(f"Current working directory: {os.getcwd()}")

print(f"\nAll collections in this client:")
for c in client.list_collections():
    coll = client.get_collection(c.name)
    print(f"  '{c.name}': {coll.count():,} entries")

Current working directory: d:\mrsha\Projects\AutoPricer\notebooks

All collections in this client:
  'cars': 68,053 entries


In [1]:
import chromadb

client = chromadb.PersistentClient(path="../cars_vectorstore")
for c in client.list_collections():
    coll = client.get_collection(c.name)
    print(f"'{c.name}': {coll.count():,} entries")

'cars': 68,053 entries


In [3]:
import chromadb

client = chromadb.PersistentClient(path="../cars_vectorstore")
collection = client.get_collection("cars")

result = collection.get(include=['documents', 'metadatas'])
new_summaries = result['documents']
new_metadatas = result['metadatas']
print(f"Recovered {len(new_summaries):,} summaries from current collection")
print(new_summaries[0])
print(new_metadatas[0])

InternalError: Error executing plan: Internal error: error returned from database: (code: 1) too many SQL variables

In [5]:
from tqdm import tqdm

BATCH = 5000
total = collection.count()
print(f"Total entries to recover: {total:,}")

new_summaries = []
new_metadatas = []

for offset in tqdm(range(0, total, BATCH)):
    result = collection.get(
        include=['documents', 'metadatas'],
        limit=BATCH,
        offset=offset
    )
    new_summaries.extend(result['documents'])
    new_metadatas.extend(result['metadatas'])

print(f"Recovered {len(new_summaries):,} summaries")
print(new_summaries[0])
print(new_metadatas[0])

Total entries to recover: 68,053


100%|██████████| 14/14 [00:01<00:00, 10.24it/s]

Recovered 68,053 summaries
Title: 2017 Audi A4 2.0T quattro Premium Sedan
Category: Sedan
Make: Audi
Description: Well-maintained, one-owner Audi A4 with no reported accidents or frame damage.
Details: 46,060 miles, Gasoline, Automatic transmission.
{'price': 25995.0, 'year': 2017, 'make': 'Audi'}


In [6]:
client.delete_collection("cars")
print("Deleted old collection")

collection = client.create_collection("cars")
print(f"Created fresh collection: {collection.count():,} entries")

Deleted old collection
Created fresh collection: 0 entries


In [7]:
import sys
sys.path.append('..')
from auto_pricer.car import Car

train, val, test = Car.from_hub("ShahidHKhan/cars_full_processed")
print(f"Loaded {len(train):,} original training cars")

Loaded 231,947 original training cars


In [8]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

BATCH_SIZE = 1000
for i in tqdm(range(0, len(train), BATCH_SIZE)):
    batch = train[i:i + BATCH_SIZE]
    documents = [car.summary for car in batch]
    vectors = encoder.encode(documents).astype(float).tolist()
    metadatas = [{"make": car.make, "year": car.year, "price": car.price} for car in batch]
    ids = [f"car_{i + j}" for j in range(len(batch))]
    collection.add(ids=ids, documents=documents, embeddings=vectors, metadatas=metadatas)

print(f"Collection now has: {collection.count():,} entries")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

100%|██████████| 232/232 [1:06:24<00:00, 17.18s/it]

Collection now has: 231,947 entries


In [9]:
BATCH_SIZE = 1000
total_new = len(new_summaries)

for i in tqdm(range(0, total_new, BATCH_SIZE)):
    doc_batch = new_summaries[i:i + BATCH_SIZE]
    meta_batch = new_metadatas[i:i + BATCH_SIZE]
    vectors = encoder.encode(doc_batch).astype(float).tolist()
    ids = [f"car_new_{i + j}" for j in range(len(doc_batch))]
    collection.add(ids=ids, documents=doc_batch, embeddings=vectors, metadatas=meta_batch)

print(f"Collection now has: {collection.count():,} entries")

100%|██████████| 69/69 [24:30<00:00, 21.32s/it]

Collection now has: 300,000 entries


In [3]:
fb_text = """2011 BMW 328i xdrive - Cars & Trucks - Florida, New York | Facebook Marketplace | Facebook
Facebook
Log In
2011 BMW 328i xdrive
$5,000
Condition
Used - Good
153xxx miles
Car needs nothing. Exterior is 8/10 Interior is 9/10. All wheel drive. Zero warning lights on the dash, all services up to date."""

In [4]:
import sys
sys.path.append('..')

from auto_pricer.agents.extraction import extract_car_details, to_car, format_summary_for_pricing

extracted = extract_car_details(fb_text)
car = to_car(extracted)
car.summary = format_summary_for_pricing(extracted)
print(car.summary)

Title: 2011 BMW 328i xdrive
Category: Vehicle
Make: BMW
Description: Car needs nothing. Exterior is 8/10 Interior is 9/10. All wheel drive. Zero warning lights on the dash, all services up to date.
Details: 153,000 miles.


In [5]:
import sys, os
sys.path.append('..')

print("cwd:", os.getcwd())  # sanity check first

from auto_pricer.agents import EnsembleAgent

ensemble = EnsembleAgent()
print(ensemble.frontier.collection.count())

cwd: d:\mrsha\Projects\AutoPricer\notebooks


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

231947
